# Feature Engineering for Regression

This notebook compares a baseline linear regression model with models trained on polynomial/interactions features and mathematical transformations. It uses the built-in scikit-learn diabetes regression dataset, so no separate data download is required.

**Target:** a quantitative disease-progression measure. **Metrics:** MAE, RMSE, and R². Lower MAE/RMSE and higher R² are better.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Load the dataset and inspect it


In [ ]:
dataset = load_diabetes(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()
print('Rows, columns:', X.shape)
display(X.head())
display(X.describe().T)
print('Missing values:', int(X.isna().sum().sum()))
print('Target summary:')
display(y.describe())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 2. Establish a baseline
Median imputation is included as a safe preprocessing step, although this dataset has no missing values. Scaling is fitted on training data only to prevent leakage.


In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'R2': r2_score(y_test, pred)
    }, model

results = []
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
row, baseline_model = evaluate_model('Baseline: scaled linear regression', baseline, X_train, X_test, y_train, y_test)
results.append(row)
pd.DataFrame(results).round(3)


## 3. Add polynomial features and interactions
Degree 2 includes squared terms and pairwise interactions. Ridge regularization helps control overfitting from the expanded feature space.


In [ ]:
poly_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])
row, poly_fitted = evaluate_model('Degree-2 polynomial + Ridge', poly_model, X_train, X_test, y_train, y_test)
results.append(row)
print('Original features:', X_train.shape[1])
print('Expanded features:', poly_fitted.named_steps['poly'].transform(X_train.iloc[:2]).shape[1])
pd.DataFrame(results).round(3)


## 4. Apply mathematical transformations
The feature values in this dataset are centered and can be negative, so a log transform is not appropriate. Instead, Yeo-Johnson power transformation supports positive and negative values and can reduce skew. It is fitted only on training data.


In [ ]:
power_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson', standardize=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])
row, power_fitted = evaluate_model('Yeo-Johnson + Ridge', power_model, X_train, X_test, y_train, y_test)
results.append(row)
pd.DataFrame(results).round(3)


## 5. Compare results
This comparison uses the same held-out test set for all models. Treat small differences cautiously: results can vary with the split and dataset. For a more robust estimate, use cross-validation and tune hyperparameters on training data only.


In [ ]:
comparison = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(comparison.round(3))

ax = comparison.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Error (lower is better)')
ax.set_title('Held-out test error by feature-engineering approach')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

ax = comparison.set_index('Model')['R2'].plot(kind='bar', figsize=(9, 4))
ax.set_ylabel('R² (higher is better)')
ax.set_title('Held-out test R² by feature-engineering approach')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

best = comparison.iloc[0]
print(f"Lowest-RMSE model on this split: {best['Model']} (RMSE={best['RMSE']:.3f}, R²={best['R2']:.3f})")


## 6. Interpretation and limitations

- **Baseline:** gives a reference point with minimal feature engineering.
- **Polynomial + interactions:** can model non-linear patterns and feature combinations, but increases dimensionality and overfitting risk; Ridge regularization is used.
- **Yeo-Johnson:** changes feature distributions while supporting negative values. It may help, hurt, or make little difference; compare the measured metrics rather than assuming improvement.
- **Interpretability:** baseline linear coefficients are easiest to explain. Polynomial coefficients are harder to interpret because they represent squares and interactions. Power-transformed features also change the meaning of coefficients.
- **Fair evaluation:** preprocessing is inside pipelines, so transformations are learned from training data only. The test set is reserved for final comparison.
- **Important:** run all cells to obtain actual metric values. Do not claim a technique improved accuracy unless the displayed results support that claim.
